# Wind Atlas quickstart — windowed read (live)

Download the **mean wind speed at 100 m** for a small bounding box from the
Global Wind Atlas and read it back. The layer is a 13.9 GB global Cloud-Optimized
GeoTIFF, but the backend reads **only the bbox window** over `/vsicurl/`, so a
small area returns in seconds — no whole-file download.

_Runs live against public figshare (no credentials)._


In [1]:
from earthlens.earthlens import EarthLens


## Download one windowed subset

A ~0.5° box near DTU (Denmark). `download()` returns the written GeoTIFF path(s).


In [2]:
paths = EarthLens(
    data_source="solar-wind-atlas",
    variables=["wind_100m"],
    lat_lim=[55.0, 55.5],
    lon_lim=[12.0, 12.5],
    path="swa_out",
).download(progress_bar=False)
paths


2026-06-27 02:15:32.027 | INFO     | earthlens.solar_wind_atlas.backend:_fetch_one_layer:256 - solar_wind_atlas wind_100m: windowed /vsicurl read https://ndownloader.figshare.com/files/17247017


2026-06-27 02:15:32 | INFO | pyramids.base.config | Logging is configured.


2026-06-27 02:15:36.968 | INFO     | earthlens.solar_wind_atlas.backend:_log_attribution:313 - solar_wind_atlas attribution: Global Wind Atlas 3.0, © Technical University of Denmark (DTU) / World Bank (ESMAP). Licensed CC-BY-4.0.


[WindowsPath('C:/gdrive/algorithms/remote-sensing/earthlens/.claude/worktrees/solar-wind-atlas/docs/examples/solar_wind_atlas/swa_out/wind_100m.tif')]

## Read the result back with pyramids

The output is an EPSG:4326 GeoTIFF of mean wind speed in m/s.


In [3]:
import numpy as np
from pyramids.dataset import Dataset

ds = Dataset.read_file(str(paths[0]))
arr = np.asarray(ds.read_array(), dtype='float64')
finite = arr[np.isfinite(arr)]
print('grid     :', ds.columns, 'x', ds.rows, 'EPSG', ds.epsg)
print('wind m/s : min %.2f  mean %.2f  max %.2f' % (finite.min(), finite.mean(), finite.max()))


grid     : 201 x 201 EPSG 4326
wind m/s : min 7.45  mean 8.65  max 9.66


## Why this is fast

`download()` opens the remote COG with GDAL `/vsicurl/` and reads only the tiles
overlapping the bbox via pyramids `Dataset.read_part` — a few hundred KB, not the
13.9 GB global file. The Global Solar Atlas layers (`ghi`, `dni`, …) instead
download their full ~2.7 GB ZIP once and crop locally, since they are not served
as range-accessible COGs.
